# MLOps Lab 4: Feature Engineering

## Aim
Understand feature engineering, create derived features using the Kidney Disease dataset, and integrate feature engineering with a Scikit-learn preprocessing and model pipeline.

## Learning Outcomes
- Explain feature engineering.
- Distinguish preprocessing from feature engineering.
- Create numerical, categorical, ratio, difference, binning, and indicator features.
- Build a custom Scikit-learn transformer.
- Integrate feature engineering with preprocessing and a model.
- Save and reuse the complete pipeline.

> **Note:** The engineered features in this lab are classroom examples for demonstrating the ML/MLOps workflow. They are not clinically validated diagnostic indicators.

## 1. Feature Engineering in the MLOps Workflow

For this course, we first learned EDA, manual preprocessing, and automated preprocessing. We now add feature engineering.

```text
Data Collection
      ↓
EDA
      ↓
Basic Data Cleaning
      ↓
Feature Engineering
      ↓
Preprocessing
      ↓
Model Training
      ↓
Evaluation
      ↓
Deployment
      ↓
Monitoring
```

Feature engineering and preprocessing can be interleaved in real projects depending on the feature being created. In this lab, we create derived features and then let the preprocessing pipeline handle missing values, encoding, and scaling.

## 2. What is Feature Engineering?

**Feature engineering** is the process of creating new features or transforming existing features so that useful information can be represented more effectively for a machine learning model.

Example:

```text
age + bp
   ↓
age_bp_ratio = age / bp
```

### Preprocessing vs Feature Engineering

| Preprocessing | Feature Engineering |
|---|---|
| Handles missing values | Creates new features |
| Encodes categories | Creates ratios/differences |
| Scales numerical values | Creates groups/bins |
| Makes data suitable for algorithms | Changes the representation to add potentially useful information |

## 3. Dataset

We continue with the **Kidney Disease dataset** from the previous Pipeline lab.

Expected file:

```text
kidney_disease.csv
```

Target column:

```text
classification
```

If an `id` column exists, it is removed because it is an identifier rather than a predictive feature.

## 4. Import Libraries

In [1]:
# Import pandas for data manipulation
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

# Import train-test split
from sklearn.model_selection import train_test_split

# Import Pipeline and ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Import preprocessing tools
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Import classes required for a custom transformer
from sklearn.base import BaseEstimator, TransformerMixin

# Import Logistic Regression
from sklearn.linear_model import LogisticRegression

# Import accuracy metric
from sklearn.metrics import accuracy_score

# Import joblib to save and load the complete pipeline
import joblib

## 5. Load the Kidney Disease Dataset

In [2]:
# Load the original Kidney Disease dataset
df = pd.read_csv("kidney_disease_cleaned.csv")

# Display the first five records
df.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


## 6. Inspect the Dataset

In [3]:
# Display the number of rows and columns
print("Dataset Shape:", df.shape)

# Display column names
print("\nColumns:")
print(df.columns.tolist())

# Display data types
print("\nData Types:")
print(df.dtypes)

Dataset Shape: (400, 26)

Columns:
['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']

Data Types:
id                  int64
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc                object
pc                 object
pcc                object
ba                 object
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv                object
wc                 object
rc                 object
htn                object
dm                 object
cad                object
appet              object
pe                 object
ane                object
classification     object
dtype: object


## 7. Separate Features and Target

In [4]:
# Make a copy so the original DataFrame is not changed
data = df.copy()

# Remove the target column from the input features
X = data.drop("classification", axis=1)

# Store the target column separately
y = data["classification"]

# Remove ID if it exists
if "id" in X.columns:
    X = X.drop("id", axis=1)

print("Feature Columns:")
print(X.columns.tolist())

Feature Columns:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


## 8. Identify Existing Feature Types

In [5]:
# Identify numerical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

# Identify categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical Columns:")
print(num_cols.tolist())

print("\nCategorical Columns:")
print(cat_cols.tolist())

Numerical Columns:
['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo']

Categorical Columns:
['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


## 9. Manual Feature Engineering: Ratio

Create an illustrative ratio:

```text
age_bp_ratio = age / bp
```

This is a classroom feature-engineering example, not a clinically validated indicator.

In [6]:
# Make a copy before creating features
X_manual = X.copy()

# Replace zero BP values with NaN to avoid division by zero
X_manual["bp"] = X_manual["bp"].replace(0, np.nan)

# Create the new ratio feature
X_manual["age_bp_ratio"] = X_manual["age"] / X_manual["bp"]

# Display the result
X_manual[["age", "bp", "age_bp_ratio"]].head()

,age,bp,age_bp_ratio
0,48.0,80.0,0.600000
1,7.0,50.0,0.140000
2,62.0,80.0,0.775000
3,48.0,70.0,0.685714
4,51.0,80.0,0.637500


## 10. Manual Feature Engineering: Difference

In [7]:
# Create an illustrative difference feature
X_manual["bu_sc_difference"] = X_manual["bu"] - X_manual["sc"]

# Display the source columns and new feature
X_manual[["bu", "sc", "bu_sc_difference"]].head()

,bu,sc,bu_sc_difference
0,36.0,1.2,34.8
1,18.0,0.8,17.2
2,53.0,1.8,51.2
3,56.0,3.8,52.2
4,26.0,1.4,24.6


## 11. Manual Feature Engineering: Binning

Binning converts a continuous value into categories.

```text
0–18    → Young
19–40   → Adult
41–60   → Middle_Age
61+     → Senior
```

In [8]:
# Create age groups
X_manual["age_group"] = pd.cut(
    X_manual["age"],
    bins=[0, 18, 40, 60, 120],
    labels=["Young", "Adult", "Middle_Age", "Senior"]
)

# Display age and age group
X_manual[["age", "age_group"]].head()

,age,age_group
0,48.0,Middle_Age
1,7.0,Young
2,62.0,Senior
3,48.0,Middle_Age
4,51.0,Middle_Age


## 12. Manual Feature Engineering: Binary Feature

In [9]:
# Create a classroom indicator feature
# 1 means BP is at or above 90; 0 means it is below 90
X_manual["high_bp_indicator"] = (
    X_manual["bp"] >= 90
).astype(int)

# Display the result
X_manual[["bp", "high_bp_indicator"]].head()

,bp,high_bp_indicator
0,80.0,0
1,50.0,0
2,80.0,0
3,70.0,0
4,80.0,0


## 13. Inspect the Engineered Features

The demonstration features are:

- `age_bp_ratio`
- `bu_sc_difference`
- `age_group`
- `high_bp_indicator`

In [10]:
# Display the newly created features
engineered_columns = [
    "age_bp_ratio",
    "bu_sc_difference",
    "age_group",
    "high_bp_indicator"
]

X_manual[engineered_columns].head()

,age_bp_ratio,bu_sc_difference,age_group,high_bp_indicator
0,0.600000,34.8,Middle_Age,0
1,0.140000,17.2,Young,0
2,0.775000,51.2,Senior,0
3,0.685714,52.2,Middle_Age,0
4,0.637500,24.6,Middle_Age,0


## 14. Why Automate Feature Engineering?

Manual feature engineering creates a consistency problem.

```text
Training Data
   ↓
Create features manually
   ↓
Train model

New Data
   ↓
Must create exactly the same features
   ↓
Predict
```

Instead, feature engineering should be placed inside the pipeline so the same logic is applied to training, testing, and future incoming data.

## 15. Create a Custom Feature Engineering Transformer

In [11]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Create the demonstration engineered features."""

    def fit(self, X, y=None):
        # These transformations do not need to learn parameters
        return self

    def transform(self, X):
        # Make a copy so the original data is not modified
        X = X.copy()

        # Replace zero BP values with NaN
        if "bp" in X.columns:
            X["bp"] = X["bp"].replace(0, np.nan)

        # Create age-to-BP ratio
        if "age" in X.columns and "bp" in X.columns:
            X["age_bp_ratio"] = X["age"] / X["bp"]

        # Create BU-SC difference
        if "bu" in X.columns and "sc" in X.columns:
            X["bu_sc_difference"] = X["bu"] - X["sc"]

        # Create age groups
        if "age" in X.columns:
            X["age_group"] = pd.cut(
                X["age"],
                bins=[0, 18, 40, 60, 120],
                labels=["Young", "Adult", "Middle_Age", "Senior"]
            )

        # Create binary BP indicator
        if "bp" in X.columns:
            X["high_bp_indicator"] = (
                X["bp"] >= 90
            ).astype(int)

        # Return the data with engineered features
        return X

## 16. Test the Custom Transformer

In [12]:
# Create the feature engineering transformer
feature_engineer = FeatureEngineer()

# Apply feature engineering to the input features
X_engineered = feature_engineer.fit_transform(X)

# Display the result
X_engineered.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,htn,dm,cad,appet,pe,ane,age_bp_ratio,bu_sc_difference,age_group,high_bp_indicator
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,yes,yes,no,good,no,no,0.600000,34.8,Middle_Age,0
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,no,no,no,good,no,no,0.140000,17.2,Young,0
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,no,yes,no,poor,no,yes,0.775000,51.2,Senior,0
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,yes,no,no,poor,yes,yes,0.685714,52.2,Middle_Age,0
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,no,no,no,good,no,no,0.637500,24.6,Middle_Age,0


## 17. Check the New Features

In [13]:
# Display the number of columns before and after feature engineering
print("Columns before feature engineering:", len(X.columns))
print("Columns after feature engineering :", len(X_engineered.columns))

# Identify newly created columns
new_features = [
    column for column in X_engineered.columns
    if column not in X.columns
]

print("\nNew Features:")
print(new_features)

Columns before feature engineering: 24
Columns after feature engineering : 28

New Features:
['age_bp_ratio', 'bu_sc_difference', 'age_group', 'high_bp_indicator']


## 18. Split the Dataset

Split before fitting the complete pipeline.

The pipeline will learn preprocessing parameters only from the training data.

In [14]:
# Split features and target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (320, 24)
X_test : (80, 24)
y_train: (320,)
y_test : (80,)


## 19. Identify Columns After Feature Engineering

The custom transformer adds numerical and categorical features. Therefore, the preprocessing column lists must describe the **engineered training data**.

In [15]:
# Create engineered training data only for discovering its column types
X_train_engineered = FeatureEngineer().fit_transform(X_train)

# Find numerical columns after feature engineering
num_cols_engineered = X_train_engineered.select_dtypes(
    include=["int64", "float64"]
).columns

# Find categorical columns after feature engineering
cat_cols_engineered = X_train_engineered.select_dtypes(
    include=["object", "category"]
).columns

print("Numerical columns:")
print(num_cols_engineered.tolist())

print("\nCategorical columns:")
print(cat_cols_engineered.tolist())

Numerical columns:
['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'age_bp_ratio', 'bu_sc_difference', 'high_bp_indicator']

Categorical columns:
['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'age_group']


## 20. Create the Numerical Preprocessing Pipeline

In [16]:
# Create numerical preprocessing steps
num_pipeline = Pipeline([
    # Fill missing numerical values using the training-data mean
    ("imputer", SimpleImputer(strategy="mean")),

    # Standardize numerical features
    ("scaler", StandardScaler())
])

## 21. Create the Categorical Preprocessing Pipeline

In [17]:
# Create categorical preprocessing steps
cat_pipeline = Pipeline([
    # Fill missing categories using the most frequent value
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Convert categories into numerical columns
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

## 22. Combine Numerical and Categorical Preprocessing

In [18]:
# Apply the appropriate preprocessing to each column type
preprocessor = ColumnTransformer([
    # Numerical columns
    ("num", num_pipeline, num_cols_engineered),

    # Categorical columns
    ("cat", cat_pipeline, cat_cols_engineered)
])

## 23. Create the Complete Pipeline

The complete pipeline is:

```text
Raw Input
   ↓
Feature Engineering
   ↓
Missing Value Imputation
   ↓
Encoding
   ↓
Scaling
   ↓
Logistic Regression
```

In [19]:
# Create one complete reusable pipeline
complete_pipeline = Pipeline([
    # Create engineered features
    ("feature_engineering", FeatureEngineer()),

    # Impute, encode, and scale the engineered data
    ("preprocessing", preprocessor),

    # Train the classification model
    ("model", LogisticRegression(max_iter=1000))
])

## 24. Train the Complete Pipeline

In [20]:
# Fit the complete pipeline using training data
complete_pipeline.fit(X_train, y_train)

print("Complete pipeline trained successfully.")

Complete pipeline trained successfully.


## 25. Evaluate the Pipeline

In [21]:
# Predict the test data
test_predictions = complete_pipeline.predict(X_test)

# Calculate accuracy
test_accuracy = accuracy_score(y_test, test_predictions)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 1.0


## 26. Save the Complete Pipeline

In [22]:
# Save feature engineering + preprocessing + model together
joblib.dump(
    complete_pipeline,
    "kidney_feature_engineering_pipeline.pkl"
)

print("Complete pipeline saved successfully.")

Complete pipeline saved successfully.


## 27. Load the Saved Pipeline

In [23]:
# Load the saved pipeline
loaded_pipeline = joblib.load(
    "kidney_feature_engineering_pipeline.pkl"
)

print("Complete pipeline loaded successfully.")

Complete pipeline loaded successfully.


## 28. Predict New Patient Data

`new_patients.csv` should contain the same raw input feature columns used for training, excluding the target column `classification`.

Students do **not** need to manually create the engineered features.

In [24]:
# Load new patient records
new_data = pd.read_csv("new_patients.csv")

# Remove ID if it exists
if "id" in new_data.columns:
    new_data = new_data.drop("id", axis=1)

# Predict using the saved complete pipeline
new_predictions = loaded_pipeline.predict(new_data)

# Display predictions
print("Predictions for new patients:")
print(new_predictions)

Predictions for new patients:
['notckd' 'ckd' 'notckd' 'ckd' 'ckd']


## 29. Key MLOps Concept

```text
TRAINING

Raw Training Data
       ↓
Feature Engineering
       ↓
Preprocessing
       ↓
Model
       ↓
Saved Complete Pipeline


NEW DATA

New Patient Data
       ↓
Same Saved Pipeline
       ↓
Feature Engineering
       ↓
Preprocessing
       ↓
Prediction
```

### Main takeaway

> Any feature created during training must be created in exactly the same way when new data is received. Putting feature engineering inside the pipeline helps maintain this consistency.

## 30. Student Exercise

1. Create one additional numerical feature.
2. Create one additional categorical/binned feature.
3. Add them to `FeatureEngineer`.
4. Retrain the complete pipeline.
5. Compare the accuracy before and after feature engineering.
6. Save the updated pipeline.
7. Test it using `new_patients.csv`.

### Questions
- Which features did you create?
- Why did you create them?
- Did the model accuracy change?
- Why must feature engineering be identical for training and new data?

## Viva Questions

1. What is feature engineering?
2. What is the difference between preprocessing and feature engineering?
3. What is a derived feature?
4. What is binning?
5. What is a binary/indicator feature?
6. Why should feature engineering be consistent between training and production?
7. What is a custom transformer?
8. Why do we use `BaseEstimator` and `TransformerMixin`?
9. What is the purpose of `fit()` and `transform()`?
10. Why should the complete pipeline be saved?